In [19]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from networkx.algorithms.community import greedy_modularity_communities, modularity
from sklearn.metrics import normalized_mutual_info_score

def create_network(small_size, n_small, large_size, p_in, p_out):
    G=nx.Graph()
    true_communities=[]
    current=0

    #маленькие сообщества
    for i in range(n_small):
        nodes=list(range(current, current+small_size))
        true_communities.append(set(nodes))
        G.add_nodes_from(nodes)

        for i in nodes:
            for j in nodes:
                if i<j and np.random.random()<p_in:
                    G.add_edge(i, j)
        current+=small_size

    #большое сообщество
    large_nodes=list(range(current, current + large_size))
    true_communities.append(set(large_nodes))
    G.add_nodes_from(large_nodes)

    for i in large_nodes:
        for j in large_nodes:
            if i<j and np.random.random()<p_in:
                G.add_edge(i, j)

    #связи между сообществами
    all_nodes=list(G.nodes())
    for i, u in enumerate(all_nodes):
        for v in all_nodes[i+1:]:
            u_comm=next(c for c in true_communities if u in c)
            v_comm=next(c for c in true_communities if v in c)
            if u_comm!=v_comm and np.random.random()<p_out:
                G.add_edge(u, v)
    return G, true_communities

def analyze_results(G, true_comms, found_comms, n_small):
    #NMI
    n_nodes=G.number_of_nodes()
    true_labels=np.zeros(n_nodes, dtype=int)
    found_labels=np.zeros(n_nodes, dtype=int)

    for i, comm in enumerate(true_comms):
        for node in comm:
            true_labels[node]=i

    for i, comm in enumerate(found_comms):
        for node in comm:
            found_labels[node]=i

    nmi=normalized_mutual_info_score(true_labels, found_labels)

    small_true=true_comms[:n_small]
    large_true=true_comms[-1]

    #в каких найденных сообществах находится большое
    small_with_large=set()
    for node in large_true:
        for idx, comm in enumerate(found_comms):
            if node in comm:
                small_with_large.add(idx)
                break

    #сколько маленьких сообществ остались как отдельные
    small_good=0
    for small_comm in small_true:
        small_with_small=set()
        for node in small_comm:
            for idx, comm in enumerate(found_comms):
                if node in comm:
                    small_with_small.add(idx)
                    break

        if not (small_with_small & small_with_large):
            small_good+=1

    united_count=n_small-small_good
    L=G.number_of_edges()
    return {
        'nmi': nmi,
        'united': united_count,
        'separate': small_good,
        'L': L,
        'resolution_limit': np.sqrt(2*L)
    }

n_trials=3
print("ПОЛНОЕ ИССЛЕДОВАНИЕ ЭФФЕКТА RESOLUTION LIMIT")

# 1. Влияние размера маленьких сообществ
print("\n1. ИССЛЕДОВАНИЕ РАЗМЕРА МАЛЕНЬКИХ СООБЩЕСТВ")
small_sizes=[3, 4, 5, 6, 7, 8, 9, 10, 12, 15]
n_small_fixed=3
large_size_fixed=40
p_in_fixed=0.9
p_out_fixed=0.01
print("│{:^16}│{:^16}│{:^16}│{:^14}│".format("Размер маленьких", "NMI (сред)", "Слилось (сред)", "√(2L) (сред)"))

for small_size in small_sizes:
    nmi_values = []
    united_values = []
    res_limit_values = []

    for trial in range(n_trials):
        G, true_comms = create_network(
            small_size=small_size,
            n_small=n_small_fixed,
            large_size=large_size_fixed,
            p_in=p_in_fixed,
            p_out=p_out_fixed)

        found_comms=list(greedy_modularity_communities(G))
        analysis=analyze_results(G, true_comms, found_comms, n_small_fixed)

        nmi_values.append(analysis['nmi'])
        united_values.append(analysis['united'])
        res_limit_values.append(analysis['resolution_limit'])

    avg_nmi=sum(nmi_values)/n_trials
    avg_united=sum(united_values)/n_trials
    avg_res=sum(res_limit_values)/n_trials
    print("│{:^16}│{:^16.3f}│{:^16.1f}│{:^14.1f}│".format(small_size, avg_nmi, avg_united, avg_res))

#2. Влияние количества маленьких сообществ
print("\n2. ИССЛЕДОВАНИЕ КОЛИЧЕСТВА МАЛЕНЬКИХ СООБЩЕСТВ")
n_small_values=[1, 2, 3, 4, 5, 6, 8, 10]
small_size_fixed=5
large_size_fixed=40
p_in_fixed=0.9
p_out_fixed=0.01
print("│{:^8}│{:^16}│{:^16}│".format("Кол-во", "NMI (сред)", "Слилось (сред)"))

for n_small in n_small_values:
    nmi_values=[]
    united_values=[]

    for trial in range(n_trials):
        G, true_comms=create_network(
            small_size=small_size_fixed,
            n_small=n_small,
            large_size=large_size_fixed,
            p_in=p_in_fixed,
            p_out=p_out_fixed)

        found_comms=list(greedy_modularity_communities(G))
        analysis=analyze_results(G, true_comms, found_comms, n_small)

        nmi_values.append(analysis['nmi'])
        united_values.append(analysis['united'])

    avg_nmi=sum(nmi_values)/n_trials
    avg_united=sum(united_values)/n_trials
    print("│{:^8}│{:^16.3f}│{:^16.1f}│".format(n_small, avg_nmi, avg_united))

#3. Влияние внутригрупповой плотности
print("\n3. ИССЛЕДОВАНИЕ ВНУТРИГРУППОВОЙ ПЛОТНОСТИ")
p_in_values=[0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.0]
small_size_fixed=5
n_small_fixed=3
large_size_fixed=40
p_out_fixed=0.01

print("│{:^8}│{:^16}│{:^16}│".format("p_in", "NMI (сред)", "Слилось (сред)"))
for p_in in p_in_values:
    nmi_values=[]
    united_values=[]

    for trial in range(n_trials):
        G, true_comms=create_network(
            small_size=small_size_fixed,
            n_small=n_small_fixed,
            large_size=large_size_fixed,
            p_in=p_in,
            p_out=p_out_fixed)

        found_comms=list(greedy_modularity_communities(G))
        analysis=analyze_results(G, true_comms, found_comms, n_small_fixed)

        nmi_values.append(analysis['nmi'])
        united_values.append(analysis['united'])

    avg_nmi=sum(nmi_values)/n_trials
    avg_united=sum(united_values)/n_trials
    print("│{:^8}│{:^16.3f}│{:^16.1f}│".format(p_in, avg_nmi, avg_united))

#4. Влияние межгрупповой плотности
print("\n4. ИССЛЕДОВАНИЕ МЕЖГРУППОВОЙ ПЛОТНОСТИ")
p_out_values=[0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2]
small_size_fixed=5
n_small_fixed=3
large_size_fixed=40
p_in_fixed=0.9
print("│{:^8}│{:^16}│{:^16}│".format("p_out", "NMI (сред)", "Слилось (сред)"))

for p_out in p_out_values:
    nmi_values=[]
    united_values=[]

    for trial in range(n_trials):
        G, true_comms=create_network(
            small_size=small_size_fixed,
            n_small=n_small_fixed,
            large_size=large_size_fixed,
            p_in=p_in_fixed,
            p_out=p_out)

        found_comms=list(greedy_modularity_communities(G))
        analysis=analyze_results(G, true_comms, found_comms, n_small_fixed)

        nmi_values.append(analysis['nmi'])
        united_values.append(analysis['united'])

    avg_nmi=sum(nmi_values)/n_trials
    avg_united=sum(united_values)/n_trials
    print("│{:^8}│{:^16.3f}│{:^16.1f}│".format(p_out, avg_nmi, avg_united))

#5. Влияние размера большого сообщества
print("\n5. ИССЛЕДОВАНИЕ РАЗМЕРА БОЛЬШОГО СООБЩЕСТВА")
large_sizes=[20, 30, 40, 50, 60, 80, 100, 150]
small_size_fixed=5
n_small_fixed=3
p_in_fixed=0.9
p_out_fixed=0.01
print("│{:^16}│{:^16}│{:^16}│{:^14}│".format("Размер большого", "NMI (сред)", "Слилось (сред)", "√(2L) (сред)"))

for large_size in large_sizes:
    nmi_values=[]
    united_values=[]
    res_limit_values=[]

    for trial in range(n_trials):
        G, true_comms=create_network(
            small_size=small_size_fixed,
            n_small=n_small_fixed,
            large_size=large_size,
            p_in=p_in_fixed,
            p_out=p_out_fixed)

        found_comms=list(greedy_modularity_communities(G))
        analysis=analyze_results(G, true_comms, found_comms, n_small_fixed)

        nmi_values.append(analysis['nmi'])
        united_values.append(analysis['united'])
        res_limit_values.append(analysis['resolution_limit'])

    avg_nmi=sum(nmi_values)/n_trials
    avg_united=sum(united_values)/n_trials
    avg_res=sum(res_limit_values)/n_trials
    print("│{:^16}│{:^16.3f}│{:^16.1f}│{:^14.1f}│".format(large_size, avg_nmi, avg_united, avg_res))

print("ИТОГОВЫЙ АНАЛИЗ ЭФФЕКТА RESOLUTION LIMIT")
print()
print("""1) при изменении размера маленьких сообществ:
Сообщества размером > 10 узлов остаются отдельными, NMI растет с увеличением размера""")
print()
print("""2) при изменении количества маленьких сообществ:
При малом количестве (1-2) сообщества обнаруживаются хорошо,при увеличении количества >3 начинается массовое слияние,
чем больше маленьких сообществ, тем сильнее эффект resolution limit""")
print()
print("""3) при изменении внутригрупповой плотности:
NMI растет с увеличением, сообщества плохо выражены""")
print()
print("""4) при изменении межгрупповой плотности:
NMI падает с увеличением, с увеличением сообщества больше сливаются""")
print()
print("""5) при изменении внутригрупповой плотности:
NMI падает с увеличением, при большом сообществе <20 узлов маленькие сообщества обнаруживаются,
при >30 узлов начинается слияние, при >60 узлов все маленькие сообщества сливаются,
√(2L) растет с размером большого сообщества, увеличивая порог resolution limit""")
print()
print("""Эффект RESOLUTION LIMIT проявляется, когда:
1. Маленькие сообщества имеют размер меньше 8-9 узлов
2. Большое сообщество имеет размер больше 60 узлов
3. Количество маленьких сообществ больше 3
4. Внутригрупповые связи сильные (p_in>0.7)
5. Межгрупповые связи слабые (p_out<0.05)
ВЛИЯНИЕ НА КАЧЕСТВО:
NMI падает с 0.8-0.9 (отличное качество) до 0.2-0.3 (плохое качество)
Маленькие сообщества полностью сливаются с большим
Алгоритм не может обнаружить реальную структуру сети""")


ПОЛНОЕ ИССЛЕДОВАНИЕ ЭФФЕКТА RESOLUTION LIMIT

1. ИССЛЕДОВАНИЕ РАЗМЕРА МАЛЕНЬКИХ СООБЩЕСТВ
│Размер маленьких│   NMI (сред)   │ Слилось (сред) │ √(2L) (сред) │
│       3        │     0.457      │      2.7       │     37.8     │
│       4        │     0.710      │      3.0       │     38.1     │
│       5        │     0.817      │      1.0       │     38.3     │
│       6        │     0.816      │      0.3       │     38.8     │
│       7        │     0.952      │      0.3       │     39.1     │
│       8        │     0.981      │      0.3       │     39.5     │
│       9        │     0.983      │      0.3       │     40.1     │
│       10       │     0.969      │      0.0       │     40.7     │
│       12       │     0.967      │      0.0       │     42.6     │
│       15       │     1.000      │      0.0       │     44.7     │

2. ИССЛЕДОВАНИЕ КОЛИЧЕСТВА МАЛЕНЬКИХ СООБЩЕСТВ
│ Кол-во │   NMI (сред)   │ Слилось (сред) │
│   1    │     0.338      │      1.0       │
│   2    │     0.698    